# Анализ поведения пользователей интернет-магазина

## О проекте

В проекте исследуется поведение пользователей интернет-магазина электроники.

Данные представлены в событийном формате: каждая строка соответствует отдельному действию пользователя с определённым товаром в конкретный момент времени.

Основные действия пользователей:

- просмотр товара;
- добавление товара в корзину;
- покупка.

Данные позволяют анализировать не только продажи, но и полный путь пользователя внутри продукта: от первого просмотра товара до покупки, повторных посещений и последующих покупок.

## Цель исследования

Изучить поведение пользователей интернет-магазина и определить закономерности, связанные с конверсией, повторными покупками, удержанием пользователей и их ценностью для бизнеса.

В рамках исследования необходимо:

- провести проверку и предобработку данных;
- изучить основные характеристики пользователей и их поведения;
- рассчитать ключевые продуктовые и коммерческие метрики;
- построить пользовательскую воронку;
- определить основные точки потери пользователей;
- сравнить поведение первых и повторных посещений;
- исследовать повторные покупки;
- провести когортный анализ;
- рассчитать retention;
- оценить накопленную ценность пользователей;
- сформулировать и проверить аналитические гипотезы;
- определить наиболее важные закономерности;
- сформулировать выводы и возможные бизнес-рекомендации.

## Основные вопросы исследования

1. Как пользователи проходят путь от просмотра товара до покупки?
2. На каком этапе воронки происходит наибольшая потеря пользователей?
3. Чем поведение покупателей отличается от поведения пользователей без покупки?
4. Отличаются ли первые и повторные посещения?
5. Как часто покупатели совершают повторные покупки?
6. Через какое время пользователи возвращаются после первой покупки?
7. Как меняется retention разных когорт?
8. Какие когорты и пользовательские сегменты создают наибольшую ценность?
9. Какие особенности пользовательского поведения связаны с более высокой вероятностью покупки?

## Ограничение LTV

Датасет содержит ограниченный период наблюдений, поэтому полный жизненный цикл клиента неизвестен.

По этой причине в проекте рассчитывается наблюдаемая накопленная ценность пользователя за доступный период — **Observed LTV**, а не прогноз полного Lifetime Value.

Данные являются наблюдательными, поэтому обнаруженные зависимости нельзя автоматически интерпретировать как причинно-следственные.

## Загрузка данных

Датасет: **eCommerce events history in electronics store** с Kaggle.

Файл `events.csv` хранится в Google Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
path = "/content/drive/MyDrive/ecommerce_analysis/data/events.csv"

df = pd.read_csv(path)
df.head()

## Первичный аудит данных

Сначала данные рассматриваются в исходном виде без очистки и преобразований.

In [ ]:
print("Строк:", df.shape[0])
print("Столбцов:", df.shape[1])

In [ ]:
df.info()

In [ ]:
df.columns.tolist()

In [ ]:
df.sample(10, random_state=42)

In [ ]:
df["event_type"].value_counts()

In [ ]:
df["event_type"].value_counts(normalize=True).mul(100).round(2)

In [ ]:
print("Пользователей:", df["user_id"].nunique())
print("Сессий по исходному user_session:", df["user_session"].nunique())
print("Товаров:", df["product_id"].nunique())
print("Категорий:", df["category_id"].nunique())
print("Брендов:", df["brand"].nunique())

In [ ]:
missing = pd.DataFrame({
    "missing": df.isna().sum(),
    "percent": df.isna().mean().mul(100).round(2)
})

missing

In [ ]:
df.duplicated().sum()

In [ ]:
df["price"].describe()

In [ ]:
print("Нулевая цена:", (df["price"] == 0).sum())
print("Отрицательная цена:", (df["price"] < 0).sum())

In [ ]:
df.nlargest(20, "price")[[
    "product_id", "category_code", "brand", "price", "event_type"
]]

In [ ]:
df["price"].quantile([0.5, 0.9, 0.95, 0.99, 0.995, 0.999, 1])

In [ ]:
df["brand"].value_counts().head(15)

In [ ]:
df["category_code"].value_counts().head(15)

In [ ]:
event_time = pd.to_datetime(df["event_time"], utc=True)

print("Начало периода:", event_time.min())
print("Конец периода:", event_time.max())

In [ ]:
summary = pd.DataFrame({
    "metric": [
        "events",
        "users",
        "sessions",
        "products",
        "categories",
        "brands",
        "duplicates"
    ],
    "value": [
        len(df),
        df["user_id"].nunique(),
        df["user_session"].nunique(),
        df["product_id"].nunique(),
        df["category_id"].nunique(),
        df["brand"].nunique(),
        df.duplicated().sum()
    ]
})

summary

### Результаты первичного аудита

В исходном датасете:

- 885 129 событий;
- 407 283 пользователя;
- 490 398 уникальных значений `user_session`;
- 53 453 товара;
- 718 категорий;
- 999 брендов;
- период наблюдений: с 24 сентября 2020 года по 28 февраля 2021 года.

По типам событий:

- `view` — 89,68%;
- `cart` — 6,10%;
- `purchase` — 4,22%.

Доля `purchase` среди событий не является пользовательской конверсией: одна строка соответствует событию, а один пользователь может совершить множество действий.

Пропуски наиболее заметны в `category_code` — 26,69% и `brand` — 23,99%. В `user_session` отсутствуют только 165 значений.

Обнаружено 655 полных дубликатов.

Распределение цены имеет длинный правый хвост: медиана составляет 65,71, 99-й перцентиль — 889,98, а максимальное значение — 64 771,06. Экстремальные значения относятся к конкретным дорогим товарам и не удаляются автоматически.

## Предобработка данных

Перед проведением анализа необходимо подготовить данные.

Пропуски в `category_code` и `brand` не являются основанием для удаления события, поскольку идентификаторы товара и пользователя сохранены. Такие значения объединяются в отдельную категорию `unknown`.

Строки без `user_session` исключаются: их доля крайне мала, а отсутствие идентификатора затрудняет анализ поведения.

Полные дубликаты удаляются.

Экстремальные значения цены сохраняются, поскольку нет оснований считать их ошибочными.

In [ ]:
df[df["user_session"].isna()]["event_type"].value_counts()

In [ ]:
df[df.duplicated(keep=False)].sort_values(
    ["user_id", "event_time"]
).head(20)

In [ ]:
df["event_time"] = pd.to_datetime(df["event_time"], utc=True)

df["category_code"] = df["category_code"].fillna("unknown")
df["brand"] = df["brand"].fillna("unknown")

df = df.dropna(subset=["user_session"])
df = df.drop_duplicates()
df = df.reset_index(drop=True)

df["date"] = df["event_time"].dt.date
df["month"] = df["event_time"].dt.strftime("%Y-%m")
df["weekday"] = df["event_time"].dt.day_name()
df["hour"] = df["event_time"].dt.hour

In [ ]:
quality = pd.DataFrame({
    "metric": [
        "rows",
        "users",
        "products",
        "duplicates",
        "missing_values"
    ],
    "value": [
        len(df),
        df["user_id"].nunique(),
        df["product_id"].nunique(),
        df.duplicated().sum(),
        df.isna().sum().sum()
    ]
})

quality

In [ ]:
removed = 885129 - len(df)

print("Удалено строк:", removed)
print("Доля удалённых:", round(removed / 885129 * 100, 3), "%")

### Результат предобработки

После очистки в датасете осталось **884 312 событий**.

Было удалено менее 0,1% исходных наблюдений: строки без идентификатора сессии и полные дубликаты.

Пропущенные значения `category_code` и `brand` были сохранены в отдельной категории `unknown`.

Экстремальные значения стоимости товаров не были автоматически признаны ошибочными и сохранены для дальнейшего анализа.

## Определение пользовательской сессии

Дополнительная проверка показала, что исходный `user_session` нельзя использовать как надёжную границу пользовательского визита.

Один идентификатор может встречаться у нескольких пользователей, а у одного пользователя сохраняться на протяжении недель или месяцев. Это приводит к нереалистичной длительности сессий.

Для дальнейшего анализа используется аналитическое определение сессии: новая сессия начинается, если между двумя последовательными действиями одного пользователя прошло более **30 минут**.

In [ ]:
df.groupby("user_session")["user_id"].nunique().max()

In [ ]:
raw_session_stats = df.groupby(["user_id", "user_session"]).agg(
    start_time=("event_time", "min"),
    end_time=("event_time", "max")
).reset_index()

raw_session_stats["duration_min"] = (
    raw_session_stats["end_time"] - raw_session_stats["start_time"]
).dt.total_seconds() / 60

raw_session_stats["duration_min"].quantile(
    [0.5, 0.75, 0.9, 0.95, 0.99, 0.999, 1]
)

In [ ]:
df = df.sort_values(["user_id", "event_time"]).reset_index(drop=True)

df["time_gap"] = (
    df.groupby("user_id")["event_time"]
    .diff()
    .dt.total_seconds()
    .div(60)
)

df["time_gap"].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
)

In [ ]:
df["new_session"] = (
    df["time_gap"].isna()
    | (df["time_gap"] > 30)
).astype(int)

df["session_number"] = (
    df.groupby("user_id")["new_session"]
    .cumsum()
)

df["analysis_session_id"] = (
    df["user_id"].astype(str)
    + "_"
    + df["session_number"].astype(str)
)

print("Аналитических сессий:", df["analysis_session_id"].nunique())

In [ ]:
session_stats = df.groupby("analysis_session_id").agg(
    user_id=("user_id", "first"),
    start_time=("event_time", "min"),
    end_time=("event_time", "max"),
    events=("event_type", "size"),
    products=("product_id", "nunique"),
    categories=("category_id", "nunique")
).reset_index()

session_events = (
    df.groupby(["analysis_session_id", "event_type"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

session_stats = session_stats.merge(
    session_events,
    on="analysis_session_id",
    how="left"
)

session_stats["duration_min"] = (
    session_stats["end_time"] - session_stats["start_time"]
).dt.total_seconds() / 60

session_stats["has_purchase"] = session_stats["purchase"] > 0

In [ ]:
session_stats[
    ["events", "products", "categories", "duration_min"]
].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
)

In [ ]:
single_event_share = session_stats["events"].eq(1).mean() * 100
session_conversion = session_stats["has_purchase"].mean() * 100

print("Сессии с одним событием:", round(single_event_share, 2), "%")
print("Сессии с покупкой:", round(session_conversion, 2), "%")

In [ ]:
user_stats = df.groupby("user_id").agg(
    events=("event_type", "size"),
    sessions=("analysis_session_id", "nunique"),
    first_event=("event_time", "min"),
    last_event=("event_time", "max")
).reset_index()

user_purchases = (
    df[df["event_type"] == "purchase"]
    .groupby("user_id")
    .size()
    .reset_index(name="purchase_events")
)

user_stats = user_stats.merge(
    user_purchases,
    on="user_id",
    how="left"
)

user_stats["purchase_events"] = (
    user_stats["purchase_events"]
    .fillna(0)
    .astype(int)
)

In [ ]:
user_stats[
    ["events", "sessions", "purchase_events"]
].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
)

In [ ]:
one_session = user_stats["sessions"].eq(1).mean() * 100
multiple_sessions = user_stats["sessions"].gt(1).mean() * 100
buyers_share = user_stats["purchase_events"].gt(0).mean() * 100

print("Одна сессия:", round(one_session, 2), "%")
print("Больше одной сессии:", round(multiple_sessions, 2), "%")
print("Пользователи с покупкой:", round(buyers_share, 2), "%")

### Результаты анализа сессий

После разбиения активности по 30-минутному правилу получено **504 630 аналитических сессий**.

- 70,1% сессий содержат только одно событие;
- 88,75% пользователей имеют только одну аналитическую сессию;
- 11,25% пользователей имеют более одной сессии;
- 4,86% сессий содержат покупку.

Эти показатели не интерпретируются как retention, поскольку пользователи впервые появляются в разные моменты времени и имеют разный доступный период для повторного посещения.

## EDA: динамика активности и монетизации

In [ ]:
purchases = df[df["event_type"] == "purchase"].copy()

purchase_prices = purchases["price"]

purchase_prices.describe(
    percentiles=[0.9, 0.95, 0.99, 0.995, 0.999]
)

In [ ]:
purchases.nlargest(10, "price")[[
    "product_id", "category_code", "brand", "price"
]]

In [ ]:
session_stats["month"] = session_stats["start_time"].dt.strftime("%Y-%m")

monthly_sessions = (
    session_stats.groupby("month")
    .size()
    .reset_index(name="sessions")
)

monthly = df.groupby("month").agg(
    events=("event_type", "size"),
    users=("user_id", "nunique")
).reset_index()

monthly_purchases = (
    purchases.groupby("month")
    .agg(
        purchase_events=("event_type", "size"),
        buyers=("user_id", "nunique"),
        revenue=("price", "sum")
    )
    .reset_index()
)

monthly = monthly.merge(
    monthly_sessions,
    on="month",
    how="left"
)

monthly = monthly.merge(
    monthly_purchases,
    on="month",
    how="left"
)

monthly["user_conversion"] = monthly["buyers"] / monthly["users"] * 100
monthly["revenue_per_user"] = monthly["revenue"] / monthly["users"]
monthly["revenue_per_buyer"] = monthly["revenue"] / monthly["buyers"]
monthly["avg_purchase_price"] = monthly["revenue"] / monthly["purchase_events"]

monthly.round(2)

In [ ]:
plt.figure(figsize=(10, 5))
sns.lineplot(data=monthly, x="month", y="users", marker="o")
plt.title("Количество активных пользователей по месяцам")
plt.xlabel("")
plt.ylabel("Пользователи")
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.lineplot(data=monthly, x="month", y="revenue", marker="o")
plt.title("Выручка по месяцам")
plt.xlabel("")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.show()

In [ ]:
daily = df.groupby("date").agg(
    events=("event_type", "size"),
    users=("user_id", "nunique"),
    sessions=("analysis_session_id", "nunique")
).reset_index()

plt.figure(figsize=(14, 5))
sns.lineplot(data=daily, x="date", y="users")
plt.title("Количество активных пользователей по дням")
plt.xlabel("")
plt.ylabel("Пользователи")
plt.show()

### Динамика монетизации

С ноября 2020 года по январь 2021 года количество активных пользователей снизилось примерно на 12%, однако количество покупателей выросло примерно на 10%.

User Conversion увеличилась с 4,67% до 5,87%, а средняя стоимость purchase-события — примерно со 104 до 179.

В результате месячная выручка выросла почти на 89% — с 787,9 тыс. до 1,49 млн.

Рост выручки происходил не за счёт увеличения аудитории. Далее исследуется изменение воронки и структуры продаж.

## Продуктовая воронка

Воронка рассматривается на нескольких уровнях. Основной итоговый вариант рассчитывается для пары `сессия × товар`, чтобы просмотр, добавление в корзину и покупка относились к одному и тому же товару.

In [ ]:
user_events = (
    df.groupby(["user_id", "event_type"])
    .size()
    .unstack(fill_value=0)
)

user_events = user_events[["view", "cart", "purchase"]]
user_flags = user_events.gt(0)

user_funnel = pd.DataFrame({
    "stage": ["View", "Cart", "Purchase"],
    "users": [
        user_flags["view"].sum(),
        user_flags["cart"].sum(),
        user_flags["purchase"].sum()
    ]
})

user_funnel["share"] = (
    user_funnel["users"] / user_funnel["users"].iloc[0] * 100
)

user_funnel

In [ ]:
session_events = (
    df.groupby(["analysis_session_id", "event_type"])
    .size()
    .unstack(fill_value=0)
)

session_events = session_events[["view", "cart", "purchase"]]
session_flags = session_events.gt(0)

session_patterns = session_flags.value_counts().reset_index(name="sessions")
session_patterns["share"] = (
    session_patterns["sessions"]
    / session_patterns["sessions"].sum()
    * 100
)

session_patterns

In [ ]:
purchase_sessions_check = session_flags[session_flags["purchase"]]

without_cart = (~purchase_sessions_check["cart"]).mean() * 100
without_view = (~purchase_sessions_check["view"]).mean() * 100

print("Покупок без cart в сессии:", round(without_cart, 2), "%")
print("Покупок без view в сессии:", round(without_view, 2), "%")

In [ ]:
product_session_events = (
    df.groupby(
        ["analysis_session_id", "product_id", "event_type"]
    )
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

product_session_events["has_view"] = product_session_events["view"] > 0
product_session_events["has_cart"] = product_session_events["cart"] > 0
product_session_events["has_purchase"] = product_session_events["purchase"] > 0

In [ ]:
product_view_to_cart = (
    (
        product_session_events["has_view"]
        & product_session_events["has_cart"]
    ).sum()
    / product_session_events["has_view"].sum()
    * 100
)

product_cart_to_purchase = (
    (
        product_session_events["has_cart"]
        & product_session_events["has_purchase"]
    ).sum()
    / product_session_events["has_cart"].sum()
    * 100
)

product_view_to_purchase = (
    (
        product_session_events["has_view"]
        & product_session_events["has_purchase"]
    ).sum()
    / product_session_events["has_view"].sum()
    * 100
)

print("View → Cart:", round(product_view_to_cart, 2), "%")
print("Cart → Purchase:", round(product_cart_to_purchase, 2), "%")
print("View → Purchase:", round(product_view_to_purchase, 2), "%")

In [ ]:
product_patterns = (
    product_session_events[
        ["has_view", "has_cart", "has_purchase"]
    ]
    .value_counts()
    .reset_index(name="product_sessions")
)

product_patterns["share"] = (
    product_patterns["product_sessions"]
    / product_patterns["product_sessions"].sum()
    * 100
)

product_patterns

In [ ]:
product_event_times = (
    df.groupby(
        ["analysis_session_id", "product_id", "event_type"]
    )["event_time"]
    .min()
    .unstack()
)

product_full_funnel = product_event_times.dropna(
    subset=["view", "cart", "purchase"]
)

product_correct_order = (
    (product_full_funnel["view"] <= product_full_funnel["cart"])
    & (product_full_funnel["cart"] <= product_full_funnel["purchase"])
)

print(
    "Правильный порядок:",
    round(product_correct_order.mean() * 100, 2),
    "%"
)

### Результаты воронки

На уровне пары `сессия × товар`:

- `View → Cart` — **7,73%**;
- `Cart → Purchase` — **47,92%**;
- `View → Purchase` — **3,94%**.

Основная относительная потеря происходит между просмотром товара и добавлением в корзину.

Для 98,79% взаимодействий, содержащих все три этапа, события происходят в ожидаемом порядке `View → Cart → Purchase`.

При этом часть покупок не содержит зарегистрированного просмотра или добавления в корзину, поэтому последовательность рассматривается как основной, но не обязательный пользовательский путь.

## Динамика воронки и анализ категорий

In [ ]:
product_info = (
    df.groupby(["analysis_session_id", "product_id"])
    .agg(
        category=("category_code", "first"),
        brand=("brand", "first"),
        price=("price", "first"),
        month=("month", "first")
    )
    .reset_index()
)

product_session_events = product_session_events.merge(
    product_info,
    on=["analysis_session_id", "product_id"],
    how="left"
)

In [ ]:
monthly_transitions = (
    product_session_events
    .groupby("month")
    .apply(
        lambda x: pd.Series({
            "view_to_cart": (
                (x["has_view"] & x["has_cart"]).sum()
                / x["has_view"].sum()
                * 100
            ),
            "cart_to_purchase": (
                (x["has_cart"] & x["has_purchase"]).sum()
                / x["has_cart"].sum()
                * 100
            ),
            "view_to_purchase": (
                (x["has_view"] & x["has_purchase"]).sum()
                / x["has_view"].sum()
                * 100
            )
        }),
        include_groups=False
    )
    .reset_index()
)

monthly_transitions.round(2)

In [ ]:
monthly_analysis = monthly.merge(
    monthly_transitions,
    on="month",
    how="left"
)

monthly_analysis[[
    "month",
    "users",
    "buyers",
    "user_conversion",
    "revenue",
    "revenue_per_user",
    "avg_purchase_price",
    "view_to_cart",
    "cart_to_purchase",
    "view_to_purchase"
]].round(2)

In [ ]:
plt.figure(figsize=(10, 5))
sns.lineplot(
    data=monthly_transitions,
    x="month",
    y="view_to_cart",
    marker="o"
)
plt.title("View → Cart по месяцам")
plt.xlabel("")
plt.ylabel("Conversion, %")
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.lineplot(
    data=monthly_transitions,
    x="month",
    y="cart_to_purchase",
    marker="o"
)
plt.title("Cart → Purchase по месяцам")
plt.xlabel("")
plt.ylabel("Conversion, %")
plt.xticks(rotation=45)
plt.show()

In [ ]:
category_volume = (
    product_session_events[
        product_session_events["has_view"]
        & (product_session_events["category"] != "unknown")
    ]
    .groupby("category")
    .size()
    .sort_values(ascending=False)
)

top_categories = category_volume.head(15).index

category_data = product_session_events[
    product_session_events["category"].isin(top_categories)
]

In [ ]:
category_funnel = (
    category_data
    .groupby("category")
    .apply(
        lambda x: pd.Series({
            "views": x["has_view"].sum(),
            "view_to_cart": (
                (x["has_view"] & x["has_cart"]).sum()
                / x["has_view"].sum()
                * 100
            ),
            "cart_to_purchase": (
                (x["has_cart"] & x["has_purchase"]).sum()
                / x["has_cart"].sum()
                * 100
            ),
            "view_to_purchase": (
                (x["has_view"] & x["has_purchase"]).sum()
                / x["has_view"].sum()
                * 100
            )
        }),
        include_groups=False
    )
    .reset_index()
)

category_funnel.sort_values(
    "view_to_purchase",
    ascending=False
).round(2)

In [ ]:
category_plot = category_funnel.sort_values(
    "view_to_purchase"
)

plt.figure(figsize=(10, 7))
sns.barplot(
    data=category_plot,
    x="view_to_purchase",
    y="category"
)
plt.title("View → Purchase для крупнейших категорий")
plt.xlabel("Conversion, %")
plt.ylabel("")
plt.show()

### Различия воронки между категориями

Воронка заметно различается между товарными категориями.

Видеокарты имеют высокий `View → Cart` — 16,35%, но `Cart → Purchase` составляет 40,23%.

Для ноутбуков `View → Cart` составляет 7,81%, а `Cart → Purchase` — 58,95%.

Особенно показательно сравнение процессоров и пылесосов: процессоры относительно часто добавляют в корзину, но слабее завершают покупку, тогда как у пылесосов добавление в корзину происходит реже, но после него конверсия в покупку выше.

Одинаковая итоговая конверсия может скрывать разные модели пользовательского поведения, поэтому место возникновения потерь следует анализировать отдельно по категориям.

## Драйвер роста выручки

In [ ]:
monthly_category = (
    purchases.groupby(["month", "category_code"])
    .agg(
        purchases=("product_id", "size"),
        buyers=("user_id", "nunique"),
        revenue=("price", "sum"),
        avg_price=("price", "mean")
    )
    .reset_index()
)

monthly_category["revenue_share"] = (
    monthly_category["revenue"]
    / monthly_category.groupby("month")["revenue"].transform("sum")
    * 100
)

In [ ]:
november_categories = (
    monthly_category[
        (monthly_category["month"] == "2020-11")
        & (monthly_category["category_code"] != "unknown")
    ]
    .sort_values("revenue", ascending=False)
    .head(10)
)

november_categories.round(2)

In [ ]:
january_categories = (
    monthly_category[
        (monthly_category["month"] == "2021-01")
        & (monthly_category["category_code"] != "unknown")
    ]
    .sort_values("revenue", ascending=False)
    .head(10)
)

january_categories.round(2)

In [ ]:
category_comparison = (
    monthly_category[
        monthly_category["month"].isin(["2020-11", "2021-01"])
        & (monthly_category["category_code"] != "unknown")
    ]
    .pivot_table(
        index="category_code",
        columns="month",
        values="revenue_share",
        fill_value=0
    )
    .reset_index()
)

category_comparison["change"] = (
    category_comparison["2021-01"]
    - category_comparison["2020-11"]
)

category_comparison.sort_values(
    "change",
    ascending=False
).head(15).round(2)

In [ ]:
nov_revenue = monthly.loc[
    monthly["month"] == "2020-11", "revenue"
].iloc[0]

jan_revenue = monthly.loc[
    monthly["month"] == "2021-01", "revenue"
].iloc[0]

nov_videocards = november_categories.loc[
    november_categories["category_code"] == "computers.components.videocards",
    "revenue"
].iloc[0]

jan_videocards = january_categories.loc[
    january_categories["category_code"] == "computers.components.videocards",
    "revenue"
].iloc[0]

total_growth = jan_revenue - nov_revenue
videocard_growth = jan_videocards - nov_videocards

print("Рост общей выручки:", round(total_growth, 2))
print("Рост выручки видеокарт:", round(videocard_growth, 2))
print(
    "Доля видеокарт в росте:",
    round(videocard_growth / total_growth * 100, 2),
    "%"
)

In [ ]:
videocards = monthly_category[
    monthly_category["category_code"]
    == "computers.components.videocards"
]

videocards[[
    "month",
    "purchases",
    "buyers",
    "revenue",
    "avg_price",
    "revenue_share"
]].round(2)

In [ ]:
plt.figure(figsize=(10, 5))
sns.lineplot(
    data=videocards,
    x="month",
    y="purchases",
    marker="o"
)
plt.title("Количество покупок видеокарт по месяцам")
plt.xlabel("")
plt.ylabel("Покупки")
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.lineplot(
    data=videocards,
    x="month",
    y="revenue_share",
    marker="o"
)
plt.title("Доля видеокарт в выручке по месяцам")
plt.xlabel("")
plt.ylabel("Доля выручки, %")
plt.xticks(rotation=45)
plt.show()

### Причина роста выручки

Рост выручки в январе был практически полностью связан с категорией `computers.components.videocards`.

С ноября по январь:

- число purchase-событий видеокарт выросло с 589 до 2618;
- количество покупателей — с 337 до 1609;
- выручка категории — с 207,6 тыс. до 1,00 млн;
- доля видеокарт в общей выручке — с 26,35% до 67,33%.

Общая выручка между ноябрём и январём выросла примерно на 700,5 тыс., тогда как выручка видеокарт увеличилась примерно на 794,5 тыс.

Рост видеокарт составляет **113,4% общего прироста**, то есть остальные категории в совокупности показали снижение.

Рост общей метрики скрывает существенное изменение структуры продаж и увеличение зависимости бизнеса от одной товарной категории.

## Цена и прохождение воронки

In [ ]:
price_data = product_session_events[
    product_session_events["has_view"]
].copy()

price_data["price_segment"] = pd.qcut(
    price_data["price"],
    4,
    labels=["Low", "Medium", "High", "Very high"]
)

price_data.groupby(
    "price_segment",
    observed=True
)["price"].agg(
    min_price="min",
    max_price="max",
    median_price="median"
).round(2)

In [ ]:
price_funnel = (
    price_data
    .groupby("price_segment", observed=True)
    .apply(
        lambda x: pd.Series({
            "views": len(x),
            "view_to_cart": (
                (x["has_view"] & x["has_cart"]).sum()
                / x["has_view"].sum()
                * 100
            ),
            "cart_to_purchase": (
                (x["has_cart"] & x["has_purchase"]).sum()
                / x["has_cart"].sum()
                * 100
            ),
            "view_to_purchase": (
                (x["has_view"] & x["has_purchase"]).sum()
                / x["has_view"].sum()
                * 100
            )
        }),
        include_groups=False
    )
    .reset_index()
)

price_funnel.round(2)

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(
    data=price_funnel,
    x="price_segment",
    y="cart_to_purchase"
)
plt.title("Cart → Purchase по ценовым сегментам")
plt.xlabel("Ценовой сегмент")
plt.ylabel("Conversion, %")
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(
    data=price_funnel,
    x="price_segment",
    y="view_to_cart"
)
plt.title("View → Cart по ценовым сегментам")
plt.xlabel("Ценовой сегмент")
plt.ylabel("Conversion, %")
plt.show()

In [ ]:
selected_categories = [
    "computers.components.videocards",
    "electronics.telephone",
    "computers.notebook",
    "computers.peripherals.printer"
]

category_price = product_session_events[
    product_session_events["has_view"]
    & product_session_events["category"].isin(selected_categories)
].copy()

category_price["price_group"] = (
    category_price.groupby("category")["price"]
    .transform(lambda x: x > x.median())
    .map({
        False: "Below median",
        True: "Above median"
    })
)

In [ ]:
category_price_funnel = (
    category_price
    .groupby(["category", "price_group"])
    .apply(
        lambda x: pd.Series({
            "views": len(x),
            "view_to_cart": (
                (x["has_view"] & x["has_cart"]).sum()
                / x["has_view"].sum()
                * 100
            ),
            "cart_to_purchase": (
                (x["has_cart"] & x["has_purchase"]).sum()
                / x["has_cart"].sum()
                * 100
            ),
            "view_to_purchase": (
                (x["has_view"] & x["has_purchase"]).sum()
                / x["has_view"].sum()
                * 100
            )
        }),
        include_groups=False
    )
    .reset_index()
)

category_price_funnel.round(2)

### Цена и прохождение воронки

На уровне всего магазина `Cart → Purchase` последовательно снижается от 55,37% для наиболее дешёвого сегмента до 41,55% для наиболее дорогого.

Однако эта зависимость не является универсальной внутри отдельных категорий.

Для телефонов, ноутбуков и принтеров товары выше медианной стоимости имеют более низкую `Cart → Purchase`. Для видеокарт наблюдается обратная картина: более дорогие товары имеют немного более высокую конверсию.

Следовательно, агрегированную связь между ценой и конверсией нельзя интерпретировать как прямое влияние цены: существенную роль играет структура товарных категорий.

## Проверка статистических гипотез

Статистические тесты используются для проверки конкретных закономерностей, найденных в ходе EDA.

### Гипотеза 1. Стоимость телефона и завершение покупки

**H₀:** конверсия `Cart → Purchase` одинакова для телефонов ниже и выше медианной стоимости категории.

**H₁:** конверсия `Cart → Purchase` различается.

Для сравнения двух долей используется z-тест пропорций.

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

In [ ]:
phone = category_price[
    (category_price["category"] == "electronics.telephone")
    & category_price["has_cart"]
].copy()

phone_stats = phone.groupby("price_group").agg(
    carts=("has_cart", "sum"),
    purchases=("has_purchase", "sum")
)

phone_stats["conversion"] = (
    phone_stats["purchases"] / phone_stats["carts"] * 100
)

phone_stats

In [ ]:
groups = ["Below median", "Above median"]

count = phone_stats.loc[groups, "purchases"].values
nobs = phone_stats.loc[groups, "carts"].values

z_stat, p_value = proportions_ztest(count, nobs)

difference = (
    phone_stats.loc["Below median", "conversion"]
    - phone_stats.loc["Above median", "conversion"]
)

print("Разница:", round(difference, 2), "п.п.")
print("z-statistic:", round(z_stat, 2))
print("p-value:", p_value)

### Результат гипотезы 1

Для телефонов ниже медианной стоимости `Cart → Purchase` составляет **57,54%**, а для телефонов выше медианы — **47,28%**.

Разница составляет **10,26 п.п.**, `p-value = 3,48 × 10⁻¹²`.

Нулевая гипотеза отвергается: наблюдаемое различие статистически значимо.

При этом результат показывает связь стоимости с поведением пользователей внутри данной категории, но не доказывает причинного влияния цены на решение о покупке.

### Первые и повторные сессии

В агрегированных данных повторные сессии имеют более высокую конверсию, чем первые. Однако такое сравнение может быть смещено составом групп, поэтому дополнительно проводится парное сравнение первой и второй сессии одних и тех же вернувшихся пользователей.

In [ ]:
session_stats = session_stats.sort_values(
    ["user_id", "start_time"]
).reset_index(drop=True)

session_stats["session_number"] = (
    session_stats.groupby("user_id").cumcount() + 1
)

session_stats["session_type"] = np.where(
    session_stats["session_number"] == 1,
    "First",
    "Returning"
)

session_type_stats = session_stats.groupby("session_type").agg(
    sessions=("analysis_session_id", "size"),
    users=("user_id", "nunique"),
    purchase_sessions=("has_purchase", "sum"),
    avg_events=("events", "mean"),
    avg_products=("products", "mean")
)

session_type_stats["conversion"] = (
    session_type_stats["purchase_sessions"]
    / session_type_stats["sessions"]
    * 100
)

session_type_stats.round(2)

In [ ]:
session_type_plot = session_type_stats.reset_index()

plt.figure(figsize=(7, 5))
sns.barplot(
    data=session_type_plot,
    x="session_type",
    y="conversion"
)
plt.title("Конверсия первой и повторных сессий")
plt.xlabel("")
plt.ylabel("Conversion, %")
plt.show()

### Гипотеза 2. Первая и вторая сессии пользователя

Для статистической проверки рассматриваются только пользователи, имеющие как минимум две сессии.

**H₀:** вероятность покупки в первой и второй сессии одинакова.

**H₁:** вероятность покупки различается.

Поскольку сравниваются бинарные результаты для одних и тех же пользователей, используется тест Мак-Немара.

In [ ]:
paired_sessions = (
    session_stats[
        session_stats["session_number"].isin([1, 2])
    ]
    .pivot(
        index="user_id",
        columns="session_number",
        values="has_purchase"
    )
    .dropna()
)

first_conversion = paired_sessions[1].mean() * 100
second_conversion = paired_sessions[2].mean() * 100

print("Первая сессия:", round(first_conversion, 2), "%")
print("Вторая сессия:", round(second_conversion, 2), "%")
print(
    "Разница:",
    round(second_conversion - first_conversion, 2),
    "п.п."
)

In [ ]:
transition_table = pd.crosstab(
    paired_sessions[1],
    paired_sessions[2]
).reindex(
    index=[False, True],
    columns=[False, True],
    fill_value=0
)

transition_table

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

result = mcnemar(
    transition_table,
    exact=False,
    correction=True
)

print("statistic:", round(result.statistic, 2))
print("p-value:", result.pvalue)

### Результат гипотезы 2

В агрегированных данных конверсия повторных сессий составляет **7,67%** против **4,18%** для первых.

Однако среди пользователей, которые действительно имеют минимум две сессии, конверсия первой сессии составляет **9,75%**, а второй — **8,54%**. Разница равна **−1,21 п.п.**

Тест Мак-Немара показал статистически значимое различие (`p < 0.001`).

Направление эффекта оказалось противоположным первоначальному агрегированному сравнению. Пользователи, которые в дальнейшем возвращаются, уже при первом посещении являются более вовлечённой аудиторией.

Следовательно, более высокая конверсия повторных сессий в агрегированных данных не доказывает, что сам повторный визит повышает вероятность покупки. Здесь проявляется эффект отбора пользователей.

## Retention и когортный анализ

Пользователи объединяются в когорты по месяцу их первого наблюдаемого посещения.

На первом этапе рассматривается **Activity Retention**: пользователь считается вернувшимся, если в соответствующем месяце зафиксировано хотя бы одно его действие.

Сентябрьская когорта требует осторожной интерпретации, поскольку данные начинаются только 24 сентября.

In [ ]:
cohort_sessions = session_stats[
    ["user_id", "start_time"]
].copy()

cohort_sessions["activity_month"] = (
    cohort_sessions["start_time"]
    .dt.tz_localize(None)
    .dt.to_period("M")
)

first_month = (
    cohort_sessions
    .groupby("user_id")["activity_month"]
    .min()
    .rename("cohort_month")
)

cohort_sessions = cohort_sessions.merge(
    first_month,
    on="user_id"
)

cohort_sessions["cohort_index"] = (
    (
        cohort_sessions["activity_month"].dt.year
        - cohort_sessions["cohort_month"].dt.year
    ) * 12
    + cohort_sessions["activity_month"].dt.month
    - cohort_sessions["cohort_month"].dt.month
)

In [ ]:
cohort_sizes = (
    cohort_sessions[
        cohort_sessions["cohort_index"] == 0
    ]
    .groupby("cohort_month")["user_id"]
    .nunique()
)

cohort_sizes

In [ ]:
plt.figure(figsize=(9, 5))
cohort_sizes.plot(kind="bar")
plt.title("Размер пользовательских когорт")
plt.xlabel("Месяц первого появления")
plt.ylabel("Пользователи")
plt.xticks(rotation=45)
plt.show()

In [ ]:
cohort_data = (
    cohort_sessions
    .groupby(["cohort_month", "cohort_index"])["user_id"]
    .nunique()
    .reset_index(name="users")
)

retention_users = cohort_data.pivot(
    index="cohort_month",
    columns="cohort_index",
    values="users"
)

retention = (
    retention_users
    .div(retention_users[0], axis=0)
    * 100
)

retention.columns = [
    f"M{i}" for i in retention.columns
]

retention.index = retention.index.astype(str)

retention.round(2)

In [ ]:
plt.figure(figsize=(10, 6))

sns.heatmap(
    retention,
    annot=True,
    fmt=".1f"
)

plt.title("Activity Retention по когортам")
plt.xlabel("Месяц жизни пользователя")
plt.ylabel("Когорта")
plt.show()

### Результаты Activity Retention

Для полноценных когорт октября–января M1 Activity Retention составляет от **2,07% до 2,76%**.

На горизонте M2 retention снижается примерно до **1%**, а затем остаётся ещё ниже.

Между полноценными когортами не наблюдается устойчивого тренда на улучшение или ухудшение M1 retention: показатель остаётся примерно на одном низком уровне.

Сентябрьская когорта показывает более высокий M1 — 6,22%, однако она сформирована только за часть месяца и напрямую не сопоставима с остальными.

Таким образом, слабая долгосрочная возвращаемость является одной из заметных особенностей пользовательского поведения.

## Purchase Retention

Activity Retention показывает повторное взаимодействие с магазином, но не показывает, продолжает ли пользователь покупать.

Для Purchase Retention пользователь относится к когорте по месяцу своей первой наблюдаемой покупки.

In [ ]:
purchase_data = df[
    df["event_type"] == "purchase"
][["user_id", "event_time"]].copy()

purchase_data["purchase_month"] = (
    purchase_data["event_time"]
    .dt.tz_localize(None)
    .dt.to_period("M")
)

first_purchase = (
    purchase_data
    .groupby("user_id")["purchase_month"]
    .min()
    .rename("purchase_cohort")
)

purchase_data = purchase_data.merge(
    first_purchase,
    on="user_id"
)

purchase_data["cohort_index"] = (
    (
        purchase_data["purchase_month"].dt.year
        - purchase_data["purchase_cohort"].dt.year
    ) * 12
    + purchase_data["purchase_month"].dt.month
    - purchase_data["purchase_cohort"].dt.month
)

In [ ]:
purchase_cohort_sizes = (
    purchase_data[
        purchase_data["cohort_index"] == 0
    ]
    .groupby("purchase_cohort")["user_id"]
    .nunique()
)

purchase_cohort_sizes

In [ ]:
purchase_cohort_data = (
    purchase_data
    .groupby(["purchase_cohort", "cohort_index"])["user_id"]
    .nunique()
    .reset_index(name="buyers")
)

purchase_retention_users = (
    purchase_cohort_data
    .pivot(
        index="purchase_cohort",
        columns="cohort_index",
        values="buyers"
    )
)

purchase_retention = (
    purchase_retention_users
    .div(purchase_retention_users[0], axis=0)
    * 100
)

purchase_retention.columns = [
    f"M{i}" for i in purchase_retention.columns
]

purchase_retention.index = (
    purchase_retention.index.astype(str)
)

purchase_retention.round(2)

In [ ]:
plt.figure(figsize=(10, 6))

sns.heatmap(
    purchase_retention,
    annot=True,
    fmt=".1f"
)

plt.title("Purchase Retention по когортам")
plt.xlabel("Месяц после первой покупки")
plt.ylabel("Когорта первой покупки")
plt.show()

### Результаты Purchase Retention

Для полноценных когорт октября–января доля покупателей, совершивших новую покупку в следующем календарном месяце, составляет от **1,60% до 2,50%**.

На горизонте M2 повторную покупку совершает уже около **0,5–0,6%** исходной когорты.

Большинство покупателей не совершает новую покупку в последующие месяцы наблюдаемого периода.

При этом месячный Purchase Retention не учитывает повторные покупки внутри того же календарного месяца, поэтому далее анализируется повторное поведение на уровне покупательских сессий.

## Повторные покупательские сессии

В датасете отсутствует `order_id`, поэтому невозможно точно определить количество отдельных заказов.

Покупательской сессией считается аналитическая сессия, содержащая хотя бы одно событие `purchase`.

Повторным покупателем в этом разделе считается пользователь, имеющий покупки как минимум в двух разных аналитических сессиях.

In [ ]:
purchase_sessions = (
    df[df["event_type"] == "purchase"]
    .groupby(["user_id", "analysis_session_id"])
    .agg(
        purchase_time=("event_time", "min"),
        revenue=("price", "sum")
    )
    .reset_index()
)

print("Покупателей:", purchase_sessions["user_id"].nunique())
print("Покупательских сессий:", len(purchase_sessions))

In [ ]:
buyer_stats = (
    purchase_sessions
    .groupby("user_id")
    .agg(
        purchase_sessions=("analysis_session_id", "nunique"),
        revenue=("revenue", "sum"),
        first_purchase=("purchase_time", "min"),
        last_purchase=("purchase_time", "max")
    )
    .reset_index()
)

buyer_stats["purchase_sessions"].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
)

In [ ]:
repeat_purchase_rate = (
    buyer_stats["purchase_sessions"].ge(2).mean() * 100
)

print(
    "Repeat Purchase Session Rate:",
    round(repeat_purchase_rate, 2),
    "%"
)

In [ ]:
buyer_stats["purchase_group"] = pd.cut(
    buyer_stats["purchase_sessions"],
    bins=[0, 1, 2, 3, float("inf")],
    labels=["1", "2", "3", "4+"]
)

purchase_distribution = (
    buyer_stats["purchase_group"]
    .value_counts()
    .sort_index()
    .reset_index()
)

purchase_distribution.columns = [
    "purchase_sessions",
    "buyers"
]

purchase_distribution["share"] = (
    purchase_distribution["buyers"]
    / purchase_distribution["buyers"].sum()
    * 100
)

purchase_distribution

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    data=purchase_distribution,
    x="purchase_sessions",
    y="share"
)

plt.title("Количество покупательских сессий на клиента")
plt.xlabel("Покупательские сессии")
plt.ylabel("Покупатели, %")
plt.show()

In [ ]:
purchase_sessions = purchase_sessions.sort_values(
    ["user_id", "purchase_time"]
)

purchase_sessions["purchase_number"] = (
    purchase_sessions.groupby("user_id").cumcount() + 1
)

first_second = (
    purchase_sessions[
        purchase_sessions["purchase_number"].isin([1, 2])
    ]
    .pivot(
        index="user_id",
        columns="purchase_number",
        values="purchase_time"
    )
    .dropna()
)

first_second["days_to_second_purchase"] = (
    first_second[2] - first_second[1]
).dt.total_seconds() / 86400

first_second["days_to_second_purchase"].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
)

In [ ]:
time_groups = pd.cut(
    first_second["days_to_second_purchase"],
    bins=[0, 1, 7, 14, 30, 60, float("inf")],
    labels=[
        "< 1 дня",
        "1–7 дней",
        "8–14 дней",
        "15–30 дней",
        "31–60 дней",
        "60+ дней"
    ],
    include_lowest=True
)

time_distribution = (
    time_groups
    .value_counts()
    .sort_index()
    .reset_index()
)

time_distribution.columns = ["period", "buyers"]

time_distribution["share"] = (
    time_distribution["buyers"]
    / time_distribution["buyers"].sum()
    * 100
)

time_distribution

In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    data=time_distribution,
    x="period",
    y="share"
)

plt.title("Время до второй покупательской сессии")
plt.xlabel("")
plt.ylabel("Покупатели, %")
plt.xticks(rotation=30)
plt.show()

In [ ]:
purchase_sessions["purchase_date"] = (
    purchase_sessions["purchase_time"].dt.date
)

purchase_days = (
    purchase_sessions
    .groupby("user_id")["purchase_date"]
    .nunique()
)

repeat_day_rate = purchase_days.ge(2).mean() * 100

print(
    "Покупатели с покупками в разные дни:",
    round(repeat_day_rate, 2),
    "%"
)

### Результаты повторного покупательского поведения

Среди 21 304 покупателей **11,53%** имеют более одной покупательской сессии за период наблюдения. 88,47% покупателей представлены только одной покупательской сессией.

Медианное время между первой и второй покупательской сессией составляет около **0,76 дня**.

- 56,6% вторых покупательских сессий происходят менее чем через сутки;
- около 83,8% — в течение первой недели;
- около 90,4% — в течение двух недель.

Высокая концентрация повторных сессий в первые сутки может отражать как реальное повторное поведение, так и особенности аналитического определения сессии. Поэтому показатель интерпретируется как **Repeat Purchase Session Rate**, а не как точный Repeat Purchase Rate по заказам.

## Revenue Cohorts и Observed LTV

Для оценки денежной ценности покупателей используется когортный анализ выручки.

Пользователь относится к когорте по месяцу первой наблюдаемой покупки.

Из-за ограниченного периода наблюдения рассчитывается **Observed LTV** — накопленная выручка на одного исходного покупателя за доступный период.

In [ ]:
revenue_data = df[
    df["event_type"] == "purchase"
][["user_id", "event_time", "price"]].copy()

revenue_data["purchase_month"] = (
    revenue_data["event_time"]
    .dt.tz_localize(None)
    .dt.to_period("M")
)

first_purchase_month = (
    revenue_data
    .groupby("user_id")["purchase_month"]
    .min()
    .rename("cohort_month")
)

revenue_data = revenue_data.merge(
    first_purchase_month,
    on="user_id"
)

revenue_data["cohort_index"] = (
    (
        revenue_data["purchase_month"].dt.year
        - revenue_data["cohort_month"].dt.year
    ) * 12
    + revenue_data["purchase_month"].dt.month
    - revenue_data["cohort_month"].dt.month
)

In [ ]:
cohort_revenue = (
    revenue_data
    .groupby(["cohort_month", "cohort_index"])["price"]
    .sum()
    .reset_index(name="revenue")
)

cohort_revenue.head(10)

In [ ]:
cohort_revenue["revenue_per_buyer"] = (
    cohort_revenue["revenue"]
    / cohort_revenue["cohort_month"]
        .map(purchase_cohort_sizes)
)

revenue_matrix = cohort_revenue.pivot(
    index="cohort_month",
    columns="cohort_index",
    values="revenue_per_buyer"
)

revenue_matrix.round(2)

In [ ]:
observed_ltv = revenue_matrix.cumsum(axis=1)

observed_ltv.columns = [
    f"M{i}" for i in observed_ltv.columns
]

observed_ltv.index = observed_ltv.index.astype(str)

observed_ltv.round(2)

In [ ]:
plt.figure(figsize=(10, 6))

sns.heatmap(
    observed_ltv,
    annot=True,
    fmt=".1f"
)

plt.title("Observed LTV по когортам")
plt.xlabel("Месяц после первой покупки")
plt.ylabel("Когорта первой покупки")
plt.show()

In [ ]:
ltv_plot = observed_ltv.T

plt.figure(figsize=(10, 6))

for cohort in ltv_plot.columns:
    plt.plot(
        ltv_plot.index,
        ltv_plot[cohort],
        marker="o",
        label=cohort
    )

plt.title("Накопленная выручка на покупателя")
plt.xlabel("Месяц жизни")
plt.ylabel("Observed LTV")
plt.legend(title="Когорта")
plt.show()

In [ ]:
ltv_summary = pd.DataFrame({
    "M0": observed_ltv["M0"],
    "last_observed": observed_ltv.apply(
        lambda row: row.dropna().iloc[-1],
        axis=1
    )
})

ltv_summary["growth_after_M0"] = (
    (ltv_summary["last_observed"] / ltv_summary["M0"] - 1) * 100
)

ltv_summary["M0_share"] = (
    ltv_summary["M0"] / ltv_summary["last_observed"] * 100
)

ltv_summary.round(2)

### Результаты Observed LTV

Наблюдаемая ценность покупателей заметно различается между когортами.

M0 увеличивается от 147,16 для октябрьской когорты до 311,02 для январской и 313,45 для февральской.

При этом рост LTV внутри одной когорты после месяца первой покупки невелик. Для полноценных когорт октября–января последующие месяцы увеличивают накопленную выручку на покупателя примерно на **3–5%**.

Около **96–97% Observed LTV** формируется уже в M0.

Более высокий Observed LTV поздних когорт нельзя автоматически интерпретировать как улучшение качества клиентской базы: ранее было показано значительное увеличение доли дорогой категории видеокарт в структуре продаж.

## Ключевые показатели

In [ ]:
total_users = df["user_id"].nunique()
total_sessions = df["analysis_session_id"].nunique()
buyers = df.loc[df["event_type"] == "purchase", "user_id"].nunique()
revenue = df.loc[df["event_type"] == "purchase", "price"].sum()

user_conversion = buyers / total_users * 100
session_conversion = session_stats["has_purchase"].mean() * 100
arpu = revenue / total_users
arppu = revenue / buyers

kpi = pd.DataFrame({
    "Метрика": [
        "Пользователи",
        "Сессии",
        "Покупатели",
        "Выручка",
        "User Conversion",
        "Session Conversion",
        "ARPU",
        "ARPPU",
        "Repeat Purchase Session Rate"
    ],
    "Значение": [
        total_users,
        total_sessions,
        buyers,
        revenue,
        user_conversion,
        session_conversion,
        arpu,
        arppu,
        repeat_purchase_rate
    ]
})

kpi["Значение"] = kpi["Значение"].round(2)

kpi

## Итоговый dashboard

In [ ]:
funnel_plot = pd.DataFrame({
    "stage": ["View", "Cart", "Purchase"],
    "conversion": [100, 7.73, 3.94]
})

plt.figure(figsize=(8, 5))

sns.barplot(
    data=funnel_plot,
    x="stage",
    y="conversion"
)

plt.title("Воронка взаимодействия с товаром")
plt.xlabel("")
plt.ylabel("% от просмотренных товаров")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

sns.lineplot(
    data=monthly,
    x="month",
    y="revenue",
    marker="o"
)

plt.title("Выручка по месяцам")
plt.xlabel("")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

sns.lineplot(
    data=videocards,
    x="month",
    y="revenue_share",
    marker="o"
)

plt.title("Доля видеокарт в выручке")
plt.xlabel("")
plt.ylabel("Доля выручки, %")
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.heatmap(
    retention,
    annot=True,
    fmt=".1f"
)

plt.title("Activity Retention по когортам")
plt.xlabel("Месяц жизни пользователя")
plt.ylabel("Когорта")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.heatmap(
    observed_ltv,
    annot=True,
    fmt=".1f"
)

plt.title("Observed LTV по когортам")
plt.xlabel("Месяц после первой покупки")
plt.ylabel("Когорта первой покупки")
plt.show()

# Итоговые выводы

### 1. Основная потеря пользователей происходит до добавления товара в корзину

На уровне одного товара внутри одной сессии конверсия `View → Cart` составляет **7,73%**, тогда как `Cart → Purchase` — **47,92%**.

Наиболее значительный drop-off наблюдается между просмотром товара и формированием выраженного намерения совершить покупку.

При этом воронка существенно различается между категориями, поэтому общая конверсия магазина скрывает разные модели пользовательского поведения.

### 2. Рост выручки произошёл без роста аудитории

С ноября 2020 года по январь 2021 года количество активных пользователей снизилось примерно на **12%**, однако месячная выручка выросла почти на **89%**.

За этот же период User Conversion увеличилась с **4,67% до 5,87%**, а средняя стоимость purchase-события выросла примерно со **104 до 179**.

Рост выручки происходил не благодаря увеличению трафика, а благодаря более эффективной монетизации существующей аудитории и изменению структуры покупок.

### 3. Главным драйвером роста стали видеокарты

Доля категории `computers.components.videocards` в общей выручке выросла с **26,35% в ноябре до 67,33% в январе** и 69,26% в феврале.

Между ноябрём и январём общая выручка магазина выросла примерно на 700,5 тыс., тогда как выручка видеокарт увеличилась примерно на 794,5 тыс.

Таким образом, рост видеокарт составляет **113,4% общего прироста**: остальные категории в совокупности показали снижение выручки.

### 4. Цена связана с прохождением воронки, но эффект зависит от категории

На уровне всего магазина `Cart → Purchase` снижается с **55,37%** для наиболее дешёвого сегмента до **41,55%** для наиболее дорогого.

Однако после анализа внутри отдельных категорий выяснилось, что такая зависимость не универсальна.

Для телефонов конверсия товаров ниже медианной стоимости составляет **57,54%**, а выше медианы — **47,28%**. Разница 10,26 п.п. статистически значима (`p < 0.001`).

Для видеокарт аналогичного снижения не наблюдается.

### 5. Возвращающиеся пользователи отличаются от аудитории в целом, но повторный визит сам по себе не объясняет рост конверсии

В агрегированных данных повторные сессии имеют более высокую конверсию: **7,67% против 4,18%**.

Однако при парном сравнении первой и второй сессии одних и тех же вернувшихся пользователей конверсия снижается с **9,75% до 8,54%**.

Тест Мак-Немара подтверждает статистически значимое различие (`p < 0.001`).

Это показывает эффект отбора: пользователи, которые в дальнейшем возвращаются, уже во время первого посещения являются более вовлечённой аудиторией.

### 6. Долгосрочная возвращаемость пользователей низкая

Для полноценных когорт октября–января M1 Activity Retention находится примерно на уровне **2–3%**.

M1 Purchase Retention составляет примерно **1,6–2,5%**.

Устойчивого улучшения retention между полноценными когортами не наблюдается.

### 7. Повторное покупательское поведение ограничено

**88,47%** покупателей имеют только одну покупательскую сессию.

Repeat Purchase Session Rate составляет **11,53%**.

Более половины вторых покупательских сессий происходят менее чем через сутки после первой, а около 84% — в течение первой недели.

### 8. Основная наблюдаемая ценность покупателя создаётся при первой покупке

Для полноценных когорт около **96–97% Observed LTV** формируется уже в M0.

Последующие месяцы увеличивают накопленную выручку на покупателя только примерно на **3–5%**.

Более высокий Observed LTV поздних когорт во многом совпадает с ростом доли дорогой категории видеокарт и не может автоматически интерпретироваться как улучшение качества клиентской базы.

# Бизнес-рекомендации

### 1. Исследовать причины низкого `View → Cart`

Основная потеря происходит до добавления товара в корзину. Для категорий с высоким трафиком и низкой конверсией стоит отдельно исследовать карточку товара, наличие, условия доставки, цену и качество информации о продукте.

### 2. Использовать разные стратегии для разных категорий

Категории с низким `View → Cart` требуют работы с этапом выбора товара. Категории с высоким добавлением в корзину и низким `Cart → Purchase` требуют анализа причин отказа уже после формирования намерения купить.

### 3. Отдельно анализировать дорогие товары

Для ряда категорий, особенно телефонов, более высокая цена связана с меньшей вероятностью завершения покупки после добавления товара в корзину. Потенциальные изменения необходимо проверять экспериментально.

### 4. Не оценивать состояние бизнеса только по общей выручке

Рост общей выручки оказался практически полностью обусловлен одной категорией. Вместе с revenue необходимо отслеживать структуру выручки по категориям и концентрацию бизнеса.

### 5. Исследовать механики повторного взаимодействия после первой покупки

Retention и рост Observed LTV после M0 находятся на низком уровне. Это делает механики повторного взаимодействия отдельным направлением для продуктовых экспериментов.

### 6. Проверять рекомендации через A/B-тесты

Наблюдательные данные позволяют находить закономерности, но не доказывают причинность. Изменения карточек товаров, коммуникаций, скидок и сценариев корзины следует проверять контролируемыми экспериментами.

## Возможные продуктовые эксперименты

### Эксперимент 1. Улучшение карточки товара

Для категорий с большим количеством просмотров и низким `View → Cart` можно протестировать изменение карточки товара.

Основная метрика:

\[
ViewToCartCR = \frac{Cart}{View}
\]

Дополнительная метрика — `View → Purchase`.

### Эксперимент 2. Работа с дорогими товарами в корзине

Для дорогих телефонов можно протестировать механизм снижения барьера покупки, например дополнительный способ оплаты или рассрочку.

Основная метрика:

\[
CartToPurchaseCR = \frac{Purchase}{Cart}
\]

Дополнительные метрики — revenue per buyer и другие guardrail-метрики при наличии соответствующих данных.

### Эксперимент 3. Повторная коммуникация после покупки

Можно протестировать персональную рекомендацию или CRM-коммуникацию после первой покупки.

Основная метрика — Repeat Purchase Rate на фиксированном временном горизонте.

Дополнительные метрики — revenue per buyer и retention.

# Ограничения исследования

- Датасет охватывает период с 24 сентября 2020 года по 28 февраля 2021 года, поэтому полный жизненный цикл пользователей неизвестен.
- История пользователей до начала периода наблюдений отсутствует. Первое наблюдаемое посещение или покупка не обязательно являются реальными первыми взаимодействиями пользователя с магазином.
- Сентябрь является неполным месяцем и не используется как полноценная когорта при сравнении retention.
- Исходный `user_session` оказался ненадёжным для определения отдельных посещений. В проекте используется аналитическое определение сессии: новая сессия начинается после перерыва более 30 минут.
- В данных отсутствует `order_id`, поэтому невозможно точно определить число отдельных заказов. Для анализа повторного поведения используются покупательские сессии.
- Отсутствует количество единиц товара в заказе, поэтому revenue рассчитывается как сумма `price` по событиям `purchase`.
- В данных отсутствуют себестоимость, маржинальность и стоимость привлечения пользователей. Observed LTV отражает выручку, а не прибыль клиента.
- Около четверти исходных событий не имеют информации о категории или бренде.
- Наблюдаемые зависимости не доказывают причинно-следственную связь. Для проверки продуктовых изменений необходимы контролируемые эксперименты.